<a href="https://colab.research.google.com/github/roly-3618/114-2_coding/blob/main/%E3%80%8CHW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_Part2_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

安裝必要的套件

In [ ]:
!pip install -U google-generativeai

In [ ]:
!pip install -q google-generativeai

In [ ]:
# import gspread # Added for self-containment
# from google.colab import auth # Added for self-containment
# from google.auth import default # Added for self-containment
# from datetime import datetime # Added for self-containment

In [ ]:
import gspread # Added for self-containment
from google.colab import auth # Added for self-containment
from google.auth import default # Added for self-containment
from datetime import datetime # Added for self-containment

In [ ]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

from google.colab import userdata
from google import genai

### 步驟 2: 導入函式庫與設定 API 金鑰

設定 Google Sheet 連線

In [ ]:
# Global variables for Google Sheet connection (re-defined here for self-containment of this test cell)
# These should ideally be defined once in cell 9f9fcf48 and that cell executed.
SHEET_URL = "https://docs.google.com/spreadsheets/d/1f0jkB_z5XxHhVhpB2PVDENNQ_PUrb-OSHE7CGpJg6G8/edit?usp=sharing"
WORKSHEET_NAME = "工作表3"
REQUIRED_COLUMNS = ["日期", "科目", "成績", "AI 摘要"] # Modified to include "AI 摘要"

_gc = None
_ws = None

def setup_gspread(sheet_url, worksheet_name):
    global _gc, _ws
    if _gc is None or _ws is None:
        print("--- 正在進行 Google Sheet 身份驗證和連線... ---")
        try:
            auth.authenticate_user()
            creds, _ = default()
            _gc = gspread.authorize(creds)
            sh = _gc.open_by_url(sheet_url)
            _ws = sh.worksheet(worksheet_name)
            print("--- Google Sheet 連線成功。---")
        except Exception as e:
            print(f"Google Sheet 連線失敗：{e}")
            _gc = None
            _ws = None

In [ ]:
# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash-lite'

# (可選) 測試 AI 模型
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI learns from data to make predictions or decisions.


### 定義 AI 摘要函式

In [ ]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        subject, score = record
        prompt_text += f"科目：{subject}, 成績：{score}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [ ]:
def process_grades_and_summary(grade_data):
    """
    處理 Gradio 介面傳入的成績，寫入 Google Sheet 並生成 AI 摘要。
    grade_data 預期是 [科目, 成績] 的列表的列表，例如：[['國文', 90], ['英文', 85]]
    """
    global _gc, _ws

    if _ws is None:
        # 如果連線失敗，嘗試重新設定 (可能在 Gradio 介面啟動後才執行)
        setup_gspread(SHEET_URL, WORKSHEET_NAME) # Modified to pass arguments
        if _ws is None:
            return "Google Sheet 未能成功連線，請檢查錯誤訊息並重試。", ""

    if not grade_data:
        return "沒有輸入任何成績，請輸入科目和成績。", ""

    # 準備寫入 Google Sheet 的成績資料，增加日期欄位
    new_grades_for_sheet = []
    today = datetime.now().strftime('%Y-%m-%d')
    for subject, grade_str in grade_data:
        try:
            grade = int(grade_str)
            new_grades_for_sheet.append([today, subject, grade])
        except ValueError:
            return f"科目 '{subject}' 的成績 '{grade_str}' 無效，成績必須是數字。", ""

    try:
        # 將新成績寫入 Google Sheet
        _ws.append_rows(new_grades_for_sheet)
        sheet_message = "成績已成功寫入 Google Sheet。\n"
    except Exception as e:
        sheet_message = f"寫入 Google Sheet 失敗：{e}\n"
        print(f"寫入 Google Sheet 失敗：{e}")
        # 即使寫入失敗，仍嘗試生成 AI 摘要

    # 準備用於 AI 摘要的資料 (只包含科目和成績)
    grades_for_ai_summary = []
    for date, subject, grade in new_grades_for_sheet:
        grades_for_ai_summary.append([subject, grade])

    # 獲取 AI 摘要
    summary = get_ai_summary(grades_for_ai_summary)

    try:
        # 尋找第一行空白列來寫入 AI 摘要
        next_row = len(_ws.col_values(1)) + 1
        _ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        _ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            # 確保不會寫入太長導致超出單元格限制 (雖然 gspread 會自動換行)
            _ws.update_cell(next_row + i, 3, line)
        sheet_message += "AI 摘要已成功寫入 Google Sheet。"
    except Exception as e:
        sheet_message += f"寫入 AI 摘要到 Google Sheet 失敗：{e}"
        print(f"寫入 AI 摘要到 Google Sheet 失敗：{e}")

    return sheet_message, summary

In [ ]:
# 確保 Google Sheet 連線已經建立或重新建立
setup_gspread(SHEET_URL, WORKSHEET_NAME)

# 準備測試資料
test_grade_data = [
    ["國文", "85"],
    ["數學", "78"],
    ["英文", "92"]
]

print("\n--- 正在執行 process_grades_and_summary 函式單元測試... ---")

sheet_status, ai_summary_output = process_grades_and_summary(test_grade_data)

print("\n--- 函式執行結果 --- ")
print(f"Google Sheet 處理狀態: {sheet_status}")
print(f"AI 摘要:\n{ai_summary_output}")

# 檢查 _ws 是否為 None，判斷 Google Sheet 是否真的連線成功
if _ws is None:
    print("\n注意：Google Sheet 工作表物件 (_ws) 仍為 None，表示連線可能仍有問題。")
else:
    print("\nGoogle Sheet 工作表物件 (_ws) 已成功初始化，連線似乎已建立。")


--- 正在執行 process_grades_and_summary 函式單元測試... ---

--- 正在呼叫 AI 模型生成摘要... ---

--- 函式執行結果 --- 
Google Sheet 處理狀態: 成績已成功寫入 Google Sheet。
AI 摘要已成功寫入 Google Sheet。
AI 摘要:
好的，這是一份根據您提供的學生成績所整理的簡單摘要與常見迷思。

---

### 學生成績摘要

*   **國文：** 85分
*   **數學：** 78分
*   **英文：** 92分

**總結：**
從這份成績來看，該生在**英文**科目表現優異，獲得了最高的成績。**國文**成績也相當不錯，屬於良好範圍。**數學**成績則相對較低，是三個科目中需要多加關注的部分。

---

### 關於此成績的常見迷思整理

以下整理了一些可能圍繞這份成績出現的常見迷思：

1.  **迷思一：英文92分就代表英文能力頂尖，不需要再加強。**
    *   **澄清：** 雖然92分是一個很高的分數，代表該生在目前的學習進度或考試中表現出色。但「頂尖」是一個相對的概念，可能與其他學生的頂尖程度、或者該科目更高層次的學習目標（例如學術寫作、流利口語等）仍有差距。此分數僅反映了當下的學習成果，持續的練習與拓展對維持和提升能力依然重要。

2.  **迷思二：數學78分就是「不及格」或「太差」，該生數學完全不行。**
    *   **澄清：** 78分通常在許多評分標準下屬於「及格」範圍，甚至可以說是「良好」的成績。雖然相較於英文和國文分數較低，但這並不代表該生「完全不行」。可能只是代表數學相對而言，需要更多的練習、理解或者不同的學習方法。這是一個可以透過努力改進的領域。

3.  **迷思三：這三個科目的成績已經能完全反映該生的學習潛力。**
    *   **澄清：** 成績是學習成果的一種展現，但它並不能完全涵蓋一個學生的所有潛力和能力。例如，學生的創造力、解決問題的能力、團隊合作精神、藝術天賦、運動能力等，都無法直接從這三科的數字成績中體現。此外，考試的結果也可能受到當天的狀態、考試形式等因素影響。

4.  **迷思四：數學78分就代表他/她不適合走理工科。**
    *   **澄清：** 職業選擇與興趣、潛力、努力方

定義 Gradio 處理函式

In [ ]:
import gradio as gr
import time # 這裡引入 time 只是為了模擬等待時間，實際使用時可刪除

# ---------------------------------------------------------
# 模擬您的兩個主要動作：寫入 Sheet 與 呼叫 AI
# (請將這裡替換成您真實的 API 程式碼)
# def write_to_sheet(data):
#     time.sleep(2) # 模擬 Google API 連線時間
#     return f"✅ 成功寫入 {len(data)} 筆資料到 Google Sheet！"

# def get_ai_summary(data):
#     time.sleep(5) # 模擬 AI 模型思考與生成的時間
#     return "這是一段 AI 生成的摘要：目前共收到多科成績，表現穩定..."
# ---------------------------------------------------------

def add_to_dataframe(subject, score, current_data):
    if not subject or not score:
        return current_data, subject, score
    cleaned_data = [row for row in current_data if any(str(cell).strip() for cell in row)]
    cleaned_data.append([subject, score])
    return cleaned_data, "", ""

def submit_and_clear(data):
    """使用 yield 分段回報進度，解決畫面卡死的問題"""
    cleaned_data = [row for row in data if any(str(cell).strip() for cell in row)]

    if not cleaned_data:
        # 如果沒資料，直接結束 (return)
        yield "錯誤：無資料", "請先輸入成績再送出！", [["", ""]]
        return

    # --- 步驟 1：通知使用者開始寫入 Sheet ---
    yield "⏳ 正在連線並寫入 Google Sheet...", "等待中...", data

    # 執行寫入 Sheet 的動作
    sheet_msg = process_grades_and_summary(cleaned_data)

    # --- 步驟 2：Sheet 寫入完成，通知使用者正在呼叫 AI ---
    # 此時畫面上的 Sheet 狀態會更新，AI 狀態會顯示正在思考
    yield sheet_msg, "🧠 Google Sheet 已更新！正在呼叫 AI 生成摘要，這可能需要 10-15 秒，請稍候...", data

    # 執行呼叫 AI 的動作
    ai_summary = get_ai_summary(cleaned_data)

    # --- 步驟 3：全部完成，顯示最終結果並清空表格 ---
    yield sheet_msg, ai_summary, [["", ""]]


# ---------------- GUI 介面設計 (與先前相同) ----------------

with gr.Blocks() as demo:
    gr.Markdown("# 🏆 進度回報版：成績輸入與 AI 摘要工具")
    gr.Markdown("程式會在執行過程中告訴您「現在做到哪一步了」，減少無謂的乾等焦慮。")

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Row():
                sub_input = gr.Textbox(label="科目", placeholder="例如：國文")
                score_input = gr.Textbox(label="成績", placeholder="例如：90")

            add_btn = gr.Button("⬇️ 新增至表格", variant="secondary")

            grade_table = gr.Dataframe(
                headers=["科目", "成績"], value=[["", ""]], datatype=["str", "str"],
                type="array", col_count=(2, "fixed"), interactive=True, label="目前累積的成績"
            )
            submit_button = gr.Button("🚀 送出整批資料", variant="primary")

        with gr.Column(scale=1):
            sheet_output = gr.Textbox(label="Google Sheet 處理狀態")
            summary_output = gr.Textbox(label="AI 摘要", lines=15)

    # 事件綁定
    add_btn.click(
        fn=add_to_dataframe,
        inputs=[sub_input, score_input, grade_table],
        outputs=[grade_table, sub_input, score_input]
    )

    submit_button.click(
        fn=submit_and_clear,
        inputs=grade_table,
        outputs=[sheet_output, summary_output, grade_table]
    )

# 加上 queue() 可以確保任務依序排隊，避免同時送出導致崩潰
demo.queue().launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6b0177b30950e4ebe0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



--- 正在呼叫 AI 模型生成摘要... ---

--- 正在呼叫 AI 模型生成摘要... ---

--- 正在呼叫 AI 模型生成摘要... ---

--- 正在呼叫 AI 模型生成摘要... ---
